In [8]:
import sys
sys.path.append('../src')

import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)

from cleaning.missing_handler import (
    report_missing, drop_rows_missing_title, fill_missing_comment,
    replace_zero_with_nan, fill_numeric_with_median, drop_high_missingness_columns
)

# Učitaj CSV iz Lab 8
df = pd.read_csv('../data/processed/analytics/products_raw.csv')
print(f"Shape: {df.shape}")
df.head(3)

Shape: (1427, 58)


,source,fetched_at,version,file_name,type,page_number,review_id,title,comment,rating,...,price,link,page_scraped,year,awards,nominations,best_picture,year_scraped,raw,preprocessed
0,reviews_page_1.json,2026-03-22 09:22:25.843,1,NaN,NaN,NaN,R3HIEYXVQFRUBV,"Powerful, Versatile & Worth Every Penny!",I absolutely love this Ninja Kitchen System! I...,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,reviews_page_1.json,2026-03-22 09:22:25.884,1,NaN,NaN,NaN,RYDWBH5RPKGBH,Powerful and versatile — love it!,The Ninja Kitchen System has been an amazing a...,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,reviews_page_1.json,2026-03-22 09:22:25.886,1,NaN,NaN,NaN,R2ST6OORNLR2CO,Great blender,"Love this blender, it works great, very durabl...",5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:

missing_report = report_missing(df)
print(missing_report)

#Saving as CSV
missing_report.to_csv('../data/processed/cleaned/missing_report.csv')
print("\nMissing report saved!")

INFO:cleaning.missing_handler:Missing value report generated for 55 columns


                 missing_count  missing_pct    dtype
variance                  1427       100.00  float64
progress                  1427       100.00  float64
remaining                 1427       100.00  float64
running_balance           1427       100.00  float64
notes                     1427       100.00  float64
status                    1427       100.00  float64
variance_pct              1427       100.00  float64
net_flow                  1427       100.00  float64
savings_rate              1427       100.00  float64
preprocessed              1422        99.65      str
text.column_1             1422        99.65      str
text.full_text            1422        99.65      str
raw                       1422        99.65      str
text.column_2             1422        99.65      str
text.paragraphs           1419        99.44      str
variant                   1416        99.23      str
text.tables               1414        99.09      str
best_picture              1407        98.60   

In [3]:
from cleaning.string_cleaner import (
    clean_title, clean_language_code, clean_comment_text,
    extract_year_from_release_date, clean_category_string
)

df_clean = df.copy()
df_clean = clean_title(df_clean)
df_clean = clean_language_code(df_clean)
df_clean = clean_comment_text(df_clean)
df_clean = extract_year_from_release_date(df_clean)
df_clean = clean_category_string(df_clean)

print("=== Title preview ===")
print(df_clean['title'].dropna().head(5))

print("\n=== Source preview ===")
print(df_clean['source'].head(5))

print("\n=== Review year preview ===")
print(df_clean['review_year'].dropna().head(5))

INFO:cleaning.string_cleaner:clean_title: 317 titles modified
INFO:cleaning.string_cleaner:clean_language_code: normalised to lowercase
INFO:cleaning.string_cleaner:clean_comment_text: comment column cleaned
INFO:cleaning.string_cleaner:extract_year_from_release_date: extracted 105 years
INFO:cleaning.string_cleaner:clean_category_string: category normalised


=== Title preview ===
0    Powerful, Versatile & Worth Every Penny!
1           Powerful and versatile — love it!
2                               Great blender
3                               Love my Ninja
4         Does It All – Smoothies to Chopping
Name: title, dtype: str

=== Source preview ===
0    reviews_page_1.json
1    reviews_page_1.json
2    reviews_page_1.json
3    reviews_page_1.json
4    reviews_page_1.json
Name: source, dtype: str

=== Review year preview ===
326    2025
327    2025
328    2025
329    2025
330    2025
Name: review_year, dtype: Int64


In [4]:
import importlib
import analytics.regex_ops as regex_module
importlib.reload(regex_module)
from analytics.regex_ops import detect_invalid_dates, extract_numeric_from_text, flag_short_comments

print("=== Invalid dates ===")
invalid_dates = detect_invalid_dates(df_clean)
print(f"Invalid dates: {invalid_dates.sum()}")

print("\n=== Extract numeric from helpful_votes ===")
numeric_votes = extract_numeric_from_text(df_clean['helpful_votes'].dropna())
print(numeric_votes.head(5))

print("\n=== Short comments ===")
short = flag_short_comments(df_clean)
print(f"Short comments: {short.sum()}")

INFO:analytics.regex_ops:detect_invalid_dates: 1427 invalid dates found
INFO:analytics.regex_ops:extract_numeric_from_text: done
INFO:analytics.regex_ops:flag_short_comments: 3 short comments found


=== Invalid dates ===
Invalid dates: 1427

=== Extract numeric from helpful_votes ===
0    8.0
1    6.0
4    6.0
5    6.0
6    2.0
Name: helpful_votes, dtype: float64

=== Short comments ===
Short comments: 3


In [9]:
# Part 5 - Deduplication
from cleaning.deduplicator import drop_exact_duplicates, drop_duplicate_ids, drop_duplicate_titles, count_duplicates

print(f"Rows before: {len(df_clean)}")

df_clean = drop_exact_duplicates(df_clean)
print(f"After exact duplicates: {len(df_clean)}")

print(f"Duplicate review_ids: {count_duplicates(df_clean, 'review_id')}")
df_clean = drop_duplicate_ids(df_clean)
print(f"After duplicate IDs: {len(df_clean)}")

df_clean = drop_duplicate_titles(df_clean)
print(f"After duplicate titles: {len(df_clean)}")

INFO:cleaning.deduplicator:drop_exact_duplicates: removed 0 rows
INFO:cleaning.deduplicator:drop_duplicate_ids (review_id): removed 0 rows
INFO:cleaning.deduplicator:drop_duplicate_titles: removed 0 rows using ['title', 'date']


Rows before: 12
After exact duplicates: 12
Duplicate review_ids: 0
After duplicate IDs: 12
After duplicate titles: 12


In [10]:
from cleaning.type_converter import convert_dates, convert_numeric_columns, convert_category_columns, memory_report

df_before = df_clean.copy()

df_clean = convert_dates(df_clean)
df_clean = convert_numeric_columns(df_clean)
df_clean = convert_category_columns(df_clean)

print("=== Updated dtypes ===")
print(df_clean[['date', 'rating', 'source', 'category']].dtypes)

print("\n=== Memory Report ===")
memory_report(df_before, df_clean)

c:\Users\merim\Desktop\E-Commerce-Product-Intelligence\notebooks\../src\cleaning\type_converter.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(
INFO:cleaning.type_converter:convert_dates: 12 rows could not be parsed (NaT)
INFO:cleaning.type_converter:convert_numeric_columns: rating -> float32
INFO:cleaning.type_converter:convert_numeric_columns: helpful_votes -> float32
INFO:cleaning.type_converter:convert_numeric_columns: budgeted -> float32
INFO:cleaning.type_converter:convert_numeric_columns: actual -> float32
INFO:cleaning.type_converter:convert_numeric_columns: page_number -> Int64
INFO:cleaning.type_converter:convert_numeric_columns: version -> Int64
INFO:cleaning.type_converter:convert_category_columns: source -> category
INFO:cleaning.type_converter:convert_category_columns: category -> category
INFO:cleani

=== Updated dtypes ===
date        datetime64[s]
rating            float32
source           category
category         category
dtype: object

=== Memory Report ===
Memory before: 0.04 MB
Memory after:  0.03 MB
Saved:         0.00 MB  (9.5%)


In [11]:
# Part 7 - Validation
from cleaning.validator import run_all_validations
run_all_validations(df_clean)

INFO:cleaning.validator:validate_rating_range: PASSED
INFO:cleaning.validator:validate_review_year_range: PASSED
INFO:cleaning.validator:validate_no_duplicate_ids (review_id): PASSED


  FAILED: validate_no_null_titles -> Found 1 null titles
  PASSED: validate_rating_range
  PASSED: validate_review_year_range
  PASSED: validate_no_duplicate_ids

Validation complete: 3 passed, 1 failed


In [12]:
# Part 8 - Cleaning Pipeline
from cleaning.clean_pipeline import run_cleaning_pipeline

df_cleaned = run_cleaning_pipeline(df, save=True)
print(f"Cleaned dataset shape: {df_cleaned.shape}")
df_cleaned.head(3)

INFO:cleaning.clean_pipeline:=== Starting cleaning pipeline: 1427 rows ===
INFO:cleaning.clean_pipeline:Step 1: drop rows missing title
INFO:cleaning.missing_handler:drop_rows_missing_title: dropped 317 rows
INFO:cleaning.clean_pipeline:Step 2: replace zero-as-missing in financial columns
INFO:cleaning.missing_handler:replace_zero_with_nan: budgeted -> replaced 0 zeros
INFO:cleaning.missing_handler:replace_zero_with_nan: actual -> replaced 0 zeros
INFO:cleaning.clean_pipeline:Step 3: drop exact duplicates
INFO:cleaning.deduplicator:drop_exact_duplicates: removed 0 rows
INFO:cleaning.clean_pipeline:Step 4: drop duplicate IDs
INFO:cleaning.deduplicator:drop_duplicate_ids (review_id): removed 1098 rows
INFO:cleaning.clean_pipeline:Step 5: clean string columns
INFO:cleaning.string_cleaner:clean_title: 0 titles modified
INFO:cleaning.string_cleaner:clean_language_code: normalised to lowercase
INFO:cleaning.string_cleaner:clean_comment_text: comment column cleaned
INFO:cleaning.clean_pipelin

  PASSED: validate_no_null_titles
  PASSED: validate_rating_range
  PASSED: validate_review_year_range
  PASSED: validate_no_duplicate_ids

Validation complete: 4 passed, 0 failed
Cleaned dataset shape: (12, 59)


,source,fetched_at,version,file_name,type,page_number,review_id,title,comment,rating,...,link,page_scraped,year,awards,nominations,best_picture,year_scraped,raw,preprocessed,review_year
0,reviews_page_1.json,2026-03-22 09:22:25.843,1,NaN,NaN,<NA>,R3HIEYXVQFRUBV,"Powerful, Versatile & Worth Every Penny!",I absolutely love this Ninja Kitchen System! I...,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
1,reviews_page_1.json,2026-03-22 09:22:25.884,1,NaN,NaN,<NA>,RYDWBH5RPKGBH,Powerful and versatile — love it!,The Ninja Kitchen System has been an amazing a...,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
2,reviews_page_1.json,2026-03-22 09:22:25.886,1,NaN,NaN,<NA>,R2ST6OORNLR2CO,Great blender,"Love this blender, it works great, very durabl...",5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
